# ADHD Pathway Discrete-Event Simulation (DES)

 > Time unit: **days**

This notebook currently implements **Phase 1 (partial pathway)** of the ADHD service model. Patients arrive over time and move through these modeled stages:

1. Screening
2. Triage
3. Assessment
4. Exit

The simulation estimates throughput, queue pressure, and time in system over a 7-day horizon.

## Real Diagram Image (Modeled Flow)

![Modeled pathway flow](figures/pathway_modeled_so_far.png)

## Source Pathway File

The source editable pathway diagram is: [figures/adhd_pathway.drawio](figures/adhd_pathway.drawio).

## Not Yet Modeled (Next Pathway Stages)

- Referral and eligibility rules
- Diagnostic decision branching
- Treatment options and review loops
- No-shows, dropout, and re-entry
- Long-term outcomes and relapse/remission

## (Flow + KPI Map) Diagram

![Flow to KPI tracking map](figures/pathway_kpi_map.png)

### Metrics Currently Tracked

- Total arrivals
- Completed patients
- Patients still in system at run end
- For each stage (screening, triage, assessment):
- Started and completed counts
- Queue length at run end
- In-service count at run end
- Maximum queue length during run
- Maximum in-service count during run
- Time in system for completed patients (average and max)

## Step 1: Import Core Libraries

In this step, we import the main Python libraries used for simulation, numerical calculations, plotting, and iteration utilities.

In [ ]:
import simpy
import numpy as np
import matplotlib.pyplot as plt
import scipy
import itertools


## Step 2: Import Probability Distributions

Here we import reusable distribution classes that generate random samples for arrivals and service times.

In [ ]:
from adhd_simpy.Model.distributions import (Exponential, 
                                            Triangular, 
                                            Uniform, 
                                            Bernoulli, 
                                            Poisson, 
                                            LogNormal)

## Step 3: Define a Trace Utility

This helper function prints event messages only when tracing is enabled, keeping output optional and controlled.

In [ ]:
def trace(msg: str) -> None:
    """Conditionally print debug output for event-level tracing.

    This helper keeps logging logic centralized so the simulation code
    can stay readable while still supporting detailed step-by-step output.

    Args:
        msg: Message describing a simulation event.
    """
    if TRACE:
        print(msg)

## Step 4: Define the Patient Process Class

This class models a single patient journey through triage and assessment, and records time in system upon completion.

In [ ]:

"""Patient entity and lifecycle process for the ADHD pathway DES."""

# ============================================================
# PATIENT PROCESS
# ============================================================

class Patient:
    """Represents one patient moving through the pathway.

    The process is: arrival -> screening -> triage -> assessment -> exit.

    Attributes:
        env: SimPy environment.
        pid: Unique patient identifier.
        system: Reference to the ADHD_Pathway system object.
    """

    def __init__(self, env, pid, system):
        self.env = env
        self.pid = pid
        self.system = system

    def process(self):
        """SimPy generator defining the patient's full journey.

        Yields:
            SimPy events for resource requests and service-time timeouts.

        Side effects:
            Updates per-stage counters and appends the patient's total time
            in system to `self.system.completed` when the patient exits.
        """

        arrival_time = self.env.now
        
        trace(f"Patient {self.pid} arrived at time {arrival_time:.2f}")

        # =========================
        # SCREENING
        # =========================
        with self.system.screening_resource.request() as req:
            yield req
            self.system.screening_started += 1

            screening_time = self.system.screening_dist.sample()
            trace(f"Patient {self.pid} started screening at time {self.env.now:.2f}, estimated duration {screening_time:.2f}")
            yield self.env.timeout(screening_time)
            self.system.screening_completed += 1
            trace(f"Patient {self.pid} completed screening at time {self.env.now:.2f}")

        # =========================
        # TRIAGE
        # =========================
        with self.system.triage_resource.request() as req:
            yield req
            self.system.triage_started += 1

            triage_time = self.system.triage_dist.sample()
            trace(f"Patient {self.pid} started triage at time {self.env.now:.2f}, estimated duration {triage_time:.2f}")
            yield self.env.timeout(triage_time)
            self.system.triage_completed += 1
            trace(f"Patient {self.pid} completed triage at time {self.env.now:.2f}")

        # =========================
        # ASSESSMENT
        # =========================
        with self.system.assessment_resource.request() as req:
            yield req
            self.system.assessment_started += 1

            assessment_time = self.system.assessment_dist.sample()
            trace(f"Patient {self.pid} started assessment at time {self.env.now:.2f}, estimated duration {assessment_time:.2f}")
            yield self.env.timeout(assessment_time)
            self.system.assessment_completed += 1
            trace(f"Patient {self.pid} completed assessment at time {self.env.now:.2f}")

        # =========================
        # EXIT
        # =========================
        total_time = self.env.now - arrival_time
        self.system.completed.append(total_time)
        self.system.completed_patients += 1



## Step 5: Define the ADHD Pathway System Class

This class coordinates arrivals, resources, and performance tracking across the full simulation.

In [ ]:

"""System-level model for the ADHD pathway discrete-event simulation."""

# ============================================================
# ADHD PATHWAY SYSTEM
# ============================================================

class ADHD_Pathway:
    """Encapsulates pathway resources, distributions, and arrival logic.

    This class receives resource objects created outside the class so
    scenario capacity can be controlled from the parameter section.

    Args:
        env: SimPy environment.
        arrival_dist: Distribution object for time between patient arrivals.
        screening_dist: Distribution object for screening service duration.
        triage_dist: Distribution object for triage service duration.
        assessment_dist: Distribution object for assessment duration.
        screening_resource: SimPy Resource for screening.
        triage_resource: SimPy Resource for triage.
        assessment_resource: SimPy Resource for assessment.
    """

    def __init__(self, env,
                 arrival_dist,
                 screening_dist,
                 triage_dist,
                 assessment_dist,
                 screening_resource,
                 triage_resource,
                 assessment_resource):

        self.env = env

        self.arrival_dist = arrival_dist
        self.screening_dist = screening_dist
        self.triage_dist = triage_dist
        self.assessment_dist = assessment_dist

        # External resources are injected for scenario control.
        self.screening_resource = screening_resource
        self.triage_resource = triage_resource
        self.assessment_resource = assessment_resource

        # Throughput and patient-level tracking.
        self.total_arrivals = 0
        self.completed_patients = 0
        self.completed = []

        # Per-resource activity counts.
        self.screening_started = 0
        self.screening_completed = 0
        self.triage_started = 0
        self.triage_completed = 0
        self.assessment_started = 0
        self.assessment_completed = 0

        # Queue and in-service statistics (max over the run).
        self.screening_max_queue = 0
        self.triage_max_queue = 0
        self.assessment_max_queue = 0
        self.screening_max_in_service = 0
        self.triage_max_in_service = 0
        self.assessment_max_in_service = 0

        # End-of-run snapshot placeholders (filled in run()).
        self.end_screening_queue = 0
        self.end_triage_queue = 0
        self.end_assessment_queue = 0
        self.end_screening_in_service = 0
        self.end_triage_in_service = 0
        self.end_assessment_in_service = 0
        self.patients_in_system_end = 0

    def _update_resource_stats(self):
        """Capture max queue/in-service values seen so far."""
        self.screening_max_queue = max(self.screening_max_queue, len(self.screening_resource.queue))
        self.triage_max_queue = max(self.triage_max_queue, len(self.triage_resource.queue))
        self.assessment_max_queue = max(self.assessment_max_queue, len(self.assessment_resource.queue))
        self.screening_max_in_service = max(self.screening_max_in_service, len(self.screening_resource.users))
        self.triage_max_in_service = max(self.triage_max_in_service, len(self.triage_resource.users))
        self.assessment_max_in_service = max(self.assessment_max_in_service, len(self.assessment_resource.users))

    # -------------------------
    # ARRIVAL PROCESS
    # -------------------------

    def arrivals(self):
        """Generate an unbounded stream of arriving patients.

        Inter-arrival times are sampled from `self.arrival_dist`.
        """

        for i in itertools.count(1):

            yield self.env.timeout(self.arrival_dist.sample())

            self.total_arrivals += 1
            patient = Patient(self.env, i, self)
            self.env.process(patient.process())

            # Update queue stats after each new arrival is introduced.
            self._update_resource_stats()

    # -------------------------
    # RUN
    # -------------------------

    def run(self, until):
        """Start arrivals, run until `until`, and store end-of-run snapshots."""
        self.env.process(self.arrivals())
        self.env.run(until=until)

        # One final update and end-state snapshot at simulation stop time.
        self._update_resource_stats()
        self.end_screening_queue = len(self.screening_resource.queue)
        self.end_triage_queue = len(self.triage_resource.queue)
        self.end_assessment_queue = len(self.assessment_resource.queue)
        self.end_screening_in_service = len(self.screening_resource.users)
        self.end_triage_in_service = len(self.triage_resource.users)
        self.end_assessment_in_service = len(self.assessment_resource.users)

        self.patients_in_system_end = self.total_arrivals - self.completed_patients



## Step 6: Set Scenario Parameters and Build the Model

In this step, we choose run length, arrival and service assumptions, define resource capacities, and construct the model instance.

In [ ]:

"""Parameterization for the 7-day ADHD pathway simulation scenario.

All time values are in days.
"""

# ============================================================
# PARAMETERS
# ============================================================

TRACE = True  # Set True for event-level debug logs.

RUN_TIME = 7  # Simulation horizon in days.

inter_arrival_time = 0.5  # Mean days between arrivals => 2 patients/day
screening_time_range = (0.03, 0.06, 0.1)  # (min, mode, max) days
triage_time_range = (0.05, 0.1, 0.2)  # (min, mode, max) days
assessment_time_mean = 2.5  # Log-space mean for LogNormal (days scale)
assessment_time_sigma = 0.8  # Log-space sigma for LogNormal

# Resource capacities (controlled outside the model class).
screening_capacity = 3
triage_capacity = 4
assessment_capacity = 4

arrival_dist = Exponential(mean=inter_arrival_time, random_seed=1)
screening_dist = Triangular(*screening_time_range, random_seed=11)
triage_dist = Triangular(*triage_time_range, random_seed=3)
assessment_dist = LogNormal(
    mean=assessment_time_mean,
    sigma=assessment_time_sigma,
    random_seed=2
)

env = simpy.Environment()

# Define resources outside the class so you can change capacity per scenario.
screening_resource = simpy.Resource(env, capacity=screening_capacity)
triage_resource = simpy.Resource(env, capacity=triage_capacity)
assessment_resource = simpy.Resource(env, capacity=assessment_capacity)

system = ADHD_Pathway(
    env,
    arrival_dist,
    screening_dist,
    triage_dist,
    assessment_dist,
    screening_resource=screening_resource,
    triage_resource=triage_resource,
    assessment_resource=assessment_resource
)



## Step 7: Run the Simulation

This cell advances the simulation clock for the selected run horizon.

In [ ]:

# ============================================================
# RUN SIMULATION
# ============================================================

system.run(until=RUN_TIME)



## Step 8: Review Results and KPIs

This final step prints patient flow, queue and resource usage metrics, and completed-patient time-in-system statistics.

In [ ]:

"""Report patient, queue, and resource KPIs from the simulation run."""

# ============================================================
# RESULTS
# ============================================================

print("===== ADHD PATHWAY DES =====")
print("Run length (days):", RUN_TIME)

print("\n--- Patient Flow ---")
print("Total arrivals:", system.total_arrivals)
print("Completed patients:", system.completed_patients)
print("Patients still in system at run end:", system.patients_in_system_end)

print("\n--- Screening Resource ---")
print("Started screening:", system.screening_started)
print("Completed screening:", system.screening_completed)
print("Queue at run end:", system.end_screening_queue)
print("In service at run end:", system.end_screening_in_service)
print("Max queue during run:", system.screening_max_queue)
print("Max in service during run:", system.screening_max_in_service)

print("\n--- Triage Resource ---")
print("Started triage:", system.triage_started)
print("Completed triage:", system.triage_completed)
print("Queue at run end:", system.end_triage_queue)
print("In service at run end:", system.end_triage_in_service)
print("Max queue during run:", system.triage_max_queue)
print("Max in service during run:", system.triage_max_in_service)

print("\n--- Assessment Resource ---")
print("Started assessment:", system.assessment_started)
print("Completed assessment:", system.assessment_completed)
print("Queue at run end:", system.end_assessment_queue)
print("In service at run end:", system.end_assessment_in_service)
print("Max queue during run:", system.assessment_max_queue)
print("Max in service during run:", system.assessment_max_in_service)

print("\n--- Time in System (Completed Patients Only) ---")
if len(system.completed) > 0:
    print("Average total time:", np.mean(system.completed))
    print("Max total time:", np.max(system.completed))
else:
    print("Average total time: N/A (no completed patients)")
    print("Max total time: N/A (no completed patients)")